# Evaluate Gene Transformer Checkpoint

Load a ScipherModel (GeneTransformerEmbedder) checkpoint and run validation on the held-out split.

**Runs on cluster** (needs SOMA database + ESM-2 embeddings).

## 1. Setup & Config

In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib.pyplot as plt

import tiledbsoma as soma
from tiledbsoma_ml import ExperimentDataset, experiment_dataloader

# Find project root by searching upward for pyproject.toml
_cwd = Path(".").resolve()
PROJECT_ROOT = _cwd
for _parent in [_cwd] + list(_cwd.parents):
    if (_parent / "pyproject.toml").exists():
        PROJECT_ROOT = _parent
        break
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")

from scipher.hierarchy import (
    load_prebuilt_hierarchy, MarginalizationLoss, WideNN,
)
from scipher.embedders.gene_transformer import GeneTransformerEmbedder

In [ ]:
# --- Config ---
# Point to the checkpoint you want to evaluate
CHECKPOINT_PATH = DATA_DIR / "checkpoint" / "transformer_2026-04-08_2026-01-29_CL0000988" / "epoch10.pt"

HIERARCHY_DATE = "2026-01-29"
ROOT_CL_ID = "CL:0000988"
SOMA_URI = "/scratch/sigbio_project_root/sigbio_project25/jingqiao/mccell-single/soma_db_homo_sapiens"
MIN_CELL_COUNT = 50
BATCH_SIZE = 512
GPU_BATCH_SIZE = 64  # Per-GPU forward pass size
SEED = 42
NUM_WORKERS = 6
MAX_VAL_CELLS = 50_000  # Cap validation size for speed; set to None for full eval

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
effective_batch_size = BATCH_SIZE * max(num_gpus, 1)

# List available checkpoints for reference
checkpoint_base = DATA_DIR / "checkpoint"
if checkpoint_base.exists():
    for run_dir in sorted(checkpoint_base.iterdir()):
        if run_dir.is_dir():
            epochs = sorted(run_dir.glob("*.pt"))
            print(f"  {run_dir.name}/  ({len(epochs)} checkpoints)")
            for ep in epochs:
                print(f"    {ep.name}")

print(f"\nLoading: {CHECKPOINT_PATH}")
print(f"Device: {device}")
print(f"Batch size: {BATCH_SIZE} (SOMA), GPU batch size: {GPU_BATCH_SIZE}/GPU, MAX_VAL_CELLS: {MAX_VAL_CELLS}")
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

## 2. Load Checkpoint & Training Loss Curves

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

print(f"Checkpoint epoch: {ckpt['epoch']}")
print(f"Config: {ckpt['config']}")
if "epoch_times" in ckpt:
    print(f"Epoch times: {['%.1fs' % t for t in ckpt['epoch_times']]}")
    print(f"Total training time: {sum(ckpt['epoch_times']):.1f}s")

# Guard: ensure this is a transformer checkpoint
assert ckpt["config"].get("embedder_class") == "GeneTransformerEmbedder", \
    f"Expected GeneTransformerEmbedder checkpoint, got: {ckpt['config'].get('embedder_class')}"

In [ ]:
loss_history = ckpt.get("loss_history", {})

def smooth(values, window=50):
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")

if loss_history:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, key, title in zip(
        axes, ["total", "leaf", "parent"],
        ["Total Loss", "Leaf Loss", "Parent Loss"],
    ):
        vals = loss_history.get(key, [])
        if not vals:
            ax.set_title(f"{title} (no data)")
            continue
        if len(vals) > 50:
            ax.plot(smooth(vals), alpha=0.8, label="smoothed")
            ax.plot(vals, alpha=0.15, color="gray", label="raw")
            ax.legend()
        else:
            ax.plot(vals, alpha=0.8)
        ax.set_title(title)
        ax.set_xlabel("Batch")
        ax.set_ylabel("Loss")

    plt.suptitle(f"Training Loss (checkpoint epoch {ckpt['epoch']})", y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"Final total loss: {loss_history['total'][-1]:.4f}")
    print(f"Final leaf loss:  {loss_history['leaf'][-1]:.4f}")
    print(f"Final parent loss: {loss_history['parent'][-1]:.4f}")
else:
    print("No loss_history in checkpoint")

## 3. Data Loading

In [ ]:
(
    mapping_dict, leaf_values, internal_values,
    marginalization_df, parent_child_df, exclusion_df,
) = load_prebuilt_hierarchy(HIERARCHY_DATE, ROOT_CL_ID)

all_cell_values = list(leaf_values) + list(internal_values)
idx_to_cl = {v: k for k, v in mapping_dict.items()}
leaf_idx_set = {mapping_dict[cl] for cl in leaf_values}

# Build marginalization tensor for internal node prediction
leaf_values_sorted = sorted(leaf_values, key=lambda k: mapping_dict[k])
internal_values_sorted = sorted(internal_values, key=lambda k: mapping_dict[k])
marginalization_tensor = torch.FloatTensor(
    marginalization_df.loc[internal_values_sorted, leaf_values_sorted].values
).to(device)

internal_sorted_to_idx = {i: mapping_dict[cl] for i, cl in enumerate(internal_values_sorted)}

print(f"Leaf types: {len(leaf_values)}, Internal: {len(internal_values)}, Total: {len(all_cell_values)}")
print(f"Marginalization tensor: {marginalization_tensor.shape}")

In [ ]:
EMB_PATH = DATA_DIR / "embeddings" / "gene_to_embedding.pkl"
with open(EMB_PATH, "rb") as f:
    gene_to_embedding = pickle.load(f)

embed_dim = next(iter(gene_to_embedding.values())).shape[0]
print(f"Gene embeddings: {len(gene_to_embedding):,} genes x {embed_dim}-dim")

In [ ]:
BIOMART_FILE = DATA_DIR / "raw" / "mart_export.txt"
df_biomart = pd.read_csv(BIOMART_FILE)
df_pc = df_biomart[df_biomart["Gene type"] == "protein_coding"].dropna(
    subset=["Gene stable ID", "Gene name"]
)
gene_list = df_pc["Gene stable ID"].unique().tolist()

ensembl_to_symbol = (
    df_pc.drop_duplicates("Gene stable ID")
    .set_index("Gene stable ID")["Gene name"]
    .to_dict()
)
print(f"Protein-coding Ensembl IDs: {len(gene_list):,}")

In [ ]:
experiment = soma.open(SOMA_URI, mode="r")

obs_df = (
    experiment.obs.read(
        value_filter='assay == "10x 3\' v3" and is_primary_data == True',
        column_names=["soma_joinid", "cell_type_ontology_term_id", "cell_type"],
    )
    .concat()
    .to_pandas()
)
print(f"Total 10x v3 primary cells: {len(obs_df):,}")

obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(all_cell_values)].copy()
print(f"Blood cells in hierarchy: {len(obs_df):,}")

type_counts = obs_df["cell_type_ontology_term_id"].value_counts()
keep_types = type_counts[type_counts >= MIN_CELL_COUNT].index
obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(keep_types)].copy()
print(
    f"After dropping < {MIN_CELL_COUNT} cells: {len(obs_df):,} cells, "
    f"{obs_df['cell_type_ontology_term_id'].nunique()} types"
)

cl_to_name = (
    obs_df.drop_duplicates("cell_type_ontology_term_id")
    .set_index("cell_type_ontology_term_id")["cell_type"]
    .to_dict()
)

# Create dataset with same split as training (same SEED)
joinids = obs_df["soma_joinid"].values
var_value_filter = f"feature_id in {gene_list}"

with experiment.axis_query(
    measurement_name="RNA",
    obs_query=soma.AxisQuery(coords=(joinids,)),
    var_query=soma.AxisQuery(value_filter=var_value_filter),
) as query:
    var_df = query.var(column_names=["feature_id", "feature_name"]).concat().to_pandas()

    ds = ExperimentDataset(
        query,
        layer_name="raw",
        obs_column_names=["cell_type_ontology_term_id"],
        batch_size=effective_batch_size,
        shuffle=True,
        seed=SEED,
    )

# Use the same 80/20 split as training to get the held-out val set
_, val_ds = ds.random_split(0.8, 0.2, seed=SEED)

from attr import evolve
val_ds = evolve(val_ds, shuffle=False)

val_loader = experiment_dataloader(val_ds, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

print(f"\nVal set: ~{val_ds.shape[0]:,} cells x {val_ds.shape[1]:,} genes")

# Build gene embedding index from var metadata
col_indices = []
gene_names_mapped = []

seen_symbols = set()
for pos, (_, row) in enumerate(var_df.iterrows()):
    ensembl_id = row["feature_id"]
    symbol = ensembl_to_symbol.get(ensembl_id, row["feature_name"])
    if symbol in gene_to_embedding and symbol not in seen_symbols:
        col_indices.append(pos)
        gene_names_mapped.append(symbol)
        seen_symbols.add(symbol)

col_indices = np.array(col_indices)

gene_embs_tensor = torch.stack(
    [torch.from_numpy(gene_to_embedding[g]) for g in gene_names_mapped]
).to(device)

print(f"Genes with embeddings: {len(col_indices):,}/{len(var_df):,} ({100*len(col_indices)/len(var_df):.1f}%)")

## 4. Load Model & Run Validation

In [ ]:
cfg = ckpt["config"]

class ScipherModel(nn.Module):
    def __init__(self, embed_dim, num_leaves, gene_embs,
                 d_model=512, n_layers=4, n_heads=8, n_cls=8, d_ff=2048, output_dim=512, dropout=0.1):
        super().__init__()
        self.embedder = GeneTransformerEmbedder(
            gene_embed_dim=embed_dim, d_model=d_model, output_dim=output_dim,
            n_layers=n_layers, n_heads=n_heads, n_cls=n_cls, d_ff=d_ff, dropout=dropout,
        )
        self.classifier = WideNN(input_dim=output_dim, output_dim=num_leaves)
        self.register_buffer("gene_embs", gene_embs)

    def forward(self, expression, mask):
        cell_embedding, attn_weights = self.embedder(
            self.gene_embs, expression, mask,
        )
        logits = self.classifier(cell_embedding)
        return logits, cell_embedding, attn_weights

model = ScipherModel(
    embed_dim=embed_dim, num_leaves=len(leaf_values),
    gene_embs=gene_embs_tensor,
    d_model=cfg.get("d_model", 512), n_layers=cfg.get("n_layers", 4),
    n_heads=cfg.get("n_heads", 8), n_cls=cfg.get("n_cls", 8),
    d_ff=cfg.get("d_ff", 2048), output_dim=cfg.get("output_dim", 512),
    dropout=cfg.get("dropout", 0.1),
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
if num_gpus > 1:
    model = nn.DataParallel(model)
    print(f"Using DataParallel across {num_gpus} GPUs")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Loaded ScipherModel (Transformer): {n_params:,} trainable parameters")

In [ ]:
loss_fn = MarginalizationLoss(
    marginalization_df, parent_child_df, exclusion_df,
    leaf_values, internal_values, mapping_dict,
    leaf_weight=cfg.get("leaf_weight", 7.0), device=device,
)

INTERNAL_THRESHOLD = 0.5

# Build reverse mapping: mapping_dict index -> internal sorted index
idx_to_internal_sorted = {}
for i, cl in enumerate(internal_values_sorted):
    idx_to_internal_sorted[mapping_dict[cl]] = i

model.eval()
leaf_preds, leaf_labels = [], []
internal_binary_preds, internal_label_indices = [], []
val_cell_embeddings = []
val_cell_types = []
val_losses = {"total": [], "leaf": [], "parent": []}
total_cells_seen = 0
total_batches = 0

gpu_chunk_size = GPU_BATCH_SIZE * max(num_gpus, 1)

with torch.no_grad():
    for X_batch, obs_batch in val_loader:
        X = torch.from_numpy(X_batch) if isinstance(X_batch, np.ndarray) else torch.from_numpy(X_batch.toarray())
        X = X.float()
        batch_size_actual = X.shape[0]

        X_mapped = X[:, col_indices]
        mask = (X_mapped > 0).to(device)
        X_log = torch.log1p(X_mapped)
        expr_sum = X_log.sum(dim=1, keepdim=True).clamp(min=1e-10)
        expression = (X_log / expr_sum).to(device)

        labels = obs_batch["cell_type_ontology_term_id"].values
        y_batch = torch.tensor(
            [mapping_dict[t] for t in labels], device=device, dtype=torch.long
        )

        # Process in GPU-friendly chunks to avoid OOM
        logits_chunks, emb_chunks = [], []
        for start in range(0, expression.shape[0], gpu_chunk_size):
            end = min(start + gpu_chunk_size, expression.shape[0])
            chunk_logits, chunk_emb, _ = model(expression[start:end], mask[start:end])
            logits_chunks.append(chunk_logits)
            emb_chunks.append(chunk_emb)
        logits = torch.cat(logits_chunks, dim=0)
        cell_emb = torch.cat(emb_chunks, dim=0)

        total_loss, loss_leafs, loss_parents = loss_fn(logits, y_batch)
        val_losses["total"].append(total_loss.item())
        val_losses["leaf"].append(loss_leafs.item())
        val_losses["parent"].append(loss_parents.item())

        leaf_pred = torch.argmax(logits, dim=1)

        leaf_probs = F.softmax(logits, dim=1)
        internal_probs = torch.matmul(leaf_probs, marginalization_tensor.T)

        is_leaf = torch.tensor(
            [y.item() in leaf_idx_set for y in y_batch], device=device
        )
        is_internal = ~is_leaf

        if is_leaf.sum() > 0:
            leaf_preds.extend(leaf_pred[is_leaf].cpu().numpy())
            leaf_labels.extend(y_batch[is_leaf].cpu().numpy())

        if is_internal.sum() > 0:
            for sample_idx in torch.where(is_internal)[0]:
                true_idx = y_batch[sample_idx].item()
                internal_pos = idx_to_internal_sorted[true_idx]
                prob = internal_probs[sample_idx, internal_pos].item()
                internal_binary_preds.append(1 if prob >= INTERNAL_THRESHOLD else 0)
                internal_label_indices.append(true_idx)

        val_cell_embeddings.append(cell_emb.cpu().numpy())
        val_cell_types.extend(
            cl_to_name.get(t, t) for t in labels
        )

        total_cells_seen += batch_size_actual
        total_batches += 1

        if total_batches % 10 == 0:
            print(f"  Batch {total_batches}: {total_cells_seen:,} cells processed")

        if MAX_VAL_CELLS is not None and total_cells_seen >= MAX_VAL_CELLS:
            print(f"  Reached MAX_VAL_CELLS={MAX_VAL_CELLS:,} cap after {total_batches} batches")
            break

leaf_preds = np.array(leaf_preds)
leaf_labels = np.array(leaf_labels)
internal_binary_preds = np.array(internal_binary_preds)
internal_label_indices = np.array(internal_label_indices)
val_embeddings = np.concatenate(val_cell_embeddings, axis=0)

print(f"\nValidation complete ({total_batches} batches, {total_cells_seen:,} cells):")
print(f"  Leaf-labeled cells:     {len(leaf_labels):,}")
print(f"  Internal-labeled cells: {len(internal_label_indices):,}")
print(f"  Total cells:            {val_embeddings.shape[0]:,}")

## 5. Validation Loss

In [ ]:
avg_val_total = np.mean(val_losses["total"])
avg_val_leaf = np.mean(val_losses["leaf"])
avg_val_parent = np.mean(val_losses["parent"])

print(f"=== Validation Loss (avg over {len(val_losses['total'])} batches) ===")
print(f"  Total:  {avg_val_total:.4f}")
print(f"  Leaf:   {avg_val_leaf:.4f}")
print(f"  Parent: {avg_val_parent:.4f}")

# Compare with final training loss
if loss_history:
    n_compare = len(val_losses["total"])
    train_tail_total = np.mean(loss_history["total"][-n_compare:])
    train_tail_leaf = np.mean(loss_history["leaf"][-n_compare:])
    train_tail_parent = np.mean(loss_history["parent"][-n_compare:])

    print(f"\n=== Train vs Val Comparison (last {n_compare} batches of training) ===")
    print(f"  {'':20s} {'Train':>10s} {'Val':>10s} {'Gap':>10s}")
    print(f"  {'Total loss':20s} {train_tail_total:10.4f} {avg_val_total:10.4f} {avg_val_total - train_tail_total:+10.4f}")
    print(f"  {'Leaf loss':20s} {train_tail_leaf:10.4f} {avg_val_leaf:10.4f} {avg_val_leaf - train_tail_leaf:+10.4f}")
    print(f"  {'Parent loss':20s} {train_tail_parent:10.4f} {avg_val_parent:10.4f} {avg_val_parent - train_tail_parent:+10.4f}")

## 6. Classification Metrics

In [ ]:
# === Leaf metrics ===
leaf_acc = (leaf_preds == leaf_labels).mean()
leaf_micro_f1 = f1_score(leaf_labels, leaf_preds, average="micro")
leaf_macro_f1 = f1_score(leaf_labels, leaf_preds, average="macro")
leaf_weighted_f1 = f1_score(leaf_labels, leaf_preds, average="weighted")

print(f"=== Leaf-labeled cells (N={len(leaf_labels):,}) ===")
print(f"  Accuracy:    {leaf_acc:.4f}")
print(f"  Micro F1:    {leaf_micro_f1:.4f}")
print(f"  Macro F1:    {leaf_macro_f1:.4f}")
print(f"  Weighted F1: {leaf_weighted_f1:.4f}")

# Per-class leaf: precision, recall, F1, count
unique_leaf_labels = sorted(set(leaf_labels))
leaf_per_class_f1 = f1_score(leaf_labels, leaf_preds, average=None, labels=unique_leaf_labels)
leaf_per_class_prec = precision_score(leaf_labels, leaf_preds, average=None, labels=unique_leaf_labels, zero_division=0)
leaf_per_class_rec = recall_score(leaf_labels, leaf_preds, average=None, labels=unique_leaf_labels, zero_division=0)

leaf_rows = []
for idx, f1_val, prec, rec in zip(unique_leaf_labels, leaf_per_class_f1, leaf_per_class_prec, leaf_per_class_rec):
    cl_id = idx_to_cl[idx]
    name = cl_to_name.get(cl_id, cl_id)
    mask_idx = leaf_labels == idx
    n = mask_idx.sum()
    leaf_rows.append({"cell_type": name, "cl_id": cl_id, "n": n,
                      "precision": prec, "recall": rec, "f1": f1_val})

df_leaf_eval = pd.DataFrame(leaf_rows).sort_values("f1")
print(f"\nPer-class (leaf), sorted by F1:")
print(df_leaf_eval.to_string(index=False, float_format="{:.3f}".format))

# === Internal metrics (threshold-based binary) ===
if len(internal_label_indices) > 0:
    internal_acc = internal_binary_preds.mean()

    print(f"\n=== Internal-labeled cells (N={len(internal_label_indices):,}, threshold={INTERNAL_THRESHOLD}) ===")
    print(f"  Accuracy: {internal_acc:.4f}")

    from collections import defaultdict
    per_node_stats = defaultdict(lambda: {"correct": 0, "total": 0})
    for true_idx, pred_binary in zip(internal_label_indices, internal_binary_preds):
        per_node_stats[true_idx]["total"] += 1
        if pred_binary == 1:
            per_node_stats[true_idx]["correct"] += 1

    internal_rows = []
    for true_idx, stats in per_node_stats.items():
        cl_id = idx_to_cl[true_idx]
        name = cl_to_name.get(cl_id, cl_id)
        node_acc = stats["correct"] / stats["total"] if stats["total"] > 0 else 0
        internal_rows.append({"cell_type": name, "cl_id": cl_id,
                              "n": stats["total"], "accuracy": node_acc})

    df_internal_eval = pd.DataFrame(internal_rows).sort_values("accuracy")
    print(f"\nPer-class (internal), sorted by accuracy:")
    print(df_internal_eval.to_string(index=False, float_format="{:.3f}".format))
else:
    print("\nNo internal-labeled cells in validation set")

In [ ]:
# === Combined Summary ===
print("=" * 60)
print(f"EVALUATION SUMMARY  (epoch {ckpt['epoch']}, GeneTransformerEmbedder)")
print("=" * 60)
if MAX_VAL_CELLS is not None and total_cells_seen >= MAX_VAL_CELLS:
    print(f"  NOTE: Capped at MAX_VAL_CELLS={MAX_VAL_CELLS:,} (set to None for full eval)")
else:
    print(f"  NOTE: Full validation set evaluated (no cap hit)")
print(f"  Leaf:     {len(leaf_labels):>7,} samples | acc {leaf_acc:.4f} | "
      f"micro-F1 {leaf_micro_f1:.4f} | macro-F1 {leaf_macro_f1:.4f} | weighted-F1 {leaf_weighted_f1:.4f}")
if len(internal_label_indices) > 0:
    print(f"  Internal: {len(internal_label_indices):>7,} samples | acc {internal_acc:.4f} | "
          f"threshold {INTERNAL_THRESHOLD}")
else:
    print(f"  Internal: no internal-labeled cells")
print(f"  Total:    {val_embeddings.shape[0]:>7,} samples evaluated")
print("=" * 60)

## 7. UMAP Visualization

In [ ]:
import scanpy as sc

sc.settings.set_figure_params(dpi=100, frameon=False)

adata_umap = sc.AnnData(
    X=np.zeros((len(val_cell_types), 1)),
    obs=pd.DataFrame({"cell_type": val_cell_types}),
)
adata_umap.obsm["X_transformer"] = val_embeddings

sc.pp.neighbors(adata_umap, use_rep="X_transformer")
sc.tl.umap(adata_umap)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
sc.pl.umap(
    adata_umap, color="cell_type", ax=ax, show=False,
    title=f"Gene Transformer Cell Embeddings (epoch {ckpt['epoch']})",
    legend_loc="on data", legend_fontsize=6, frameon=False,
)

plt.tight_layout()
plt.savefig(
    DATA_DIR / "reports" / "transformer_eval_embedding_umap.png",
    dpi=150, bbox_inches="tight",
)
plt.show()
print("Saved to data/reports/transformer_eval_embedding_umap.png")